# Cipher Breaker 🔐
A secret message in Danish has been encrypted with a **substitution cipher** (every letter replaced by another): 29! ≈ 10³¹ possible keys. Crack it with a **language model** trained on your own IT-ret notes (how Danish does this text look?) and **simulated annealing** (swap two letters in the key, keep it if the text looks more Danish, sometimes keep it anyway to escape local optima).

Background: [[NLP Overview]], [[Probability for ML]] · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import pathlib, re, numpy as np
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
ALPHA = "abcdefghijklmnopqrstuvwxyzæøå"; IDX = {c: i for i, c in enumerate(ALPHA + " ")}
corpus = " ".join(f.read_text(encoding="utf-8").lower() for f in pathlib.Path("../../9-semester/IT-lov").rglob("*.md"))
corpus = re.sub(f"[^{ALPHA}]+", " ", corpus)
counts = np.ones((30, 30))                                     # bigram counts over letters + space, add-one smoothed
for a, c in zip(corpus, corpus[1:]): counts[IDX[a], IDX[c]] += 1
LOGP = np.log(counts / counts.sum(1, keepdims=True))
def encrypt(text, key): return text.translate(str.maketrans(ALPHA, "".join(key)))
def decrypt(text, key): return text.translate(str.maketrans("".join(key), ALPHA))
SECRET = ("den dataansvarlige skal kunne påvise at behandlingen af personoplysninger er lovlig rimelig og gennemsigtig "
          "og at oplysningerne kun indsamles til udtrykkeligt angivne og legitime formål samt at de ikke opbevares længere end nødvendigt")
true_key = list(ALPHA); np.random.default_rng(42).shuffle(true_key)
CIPHER = encrypt(SECRET, true_key); print("intercepted message:\n", CIPHER)

## 1. How Danish does it look?
Score a text by its bigram log-likelihood under the language model: $\sum_t \log P(c_{t+1}\mid c_t)$ using `LOGP` and `IDX` (letters and space). Higher = more Danish-looking.

In [ ]:
def score(text):
    # TODO 1: sum of bigram log-probabilities
    raise NotImplementedError  # TODO 1

## 2. Simulated annealing over keys\nStart from a random key (a shuffled alphabet). Each iteration swap two letters; accept if the score improves, or with probability $e^{\Delta/T}$ if it gets worse; lower the temperature linearly from `T0` to ~0. Keep track of the best key. Return `(best_key, best_score)`.

In [ ]:
def anneal(cipher, iters=20000, T0=10.0, seed=0):
    # TODO 2: swap moves, Metropolis acceptance, linear cooling, remember the best
    raise NotImplementedError  # TODO 2

In [ ]:
check("language score", lambda: score("den dataansvarlige") > score("xqz wvøjkåææpzyqcx") and abs(score("ab") - LOGP[IDX["a"], IDX["b"]]) < 1e-12,
      "Real Danish must score higher than gibberish; score('ab') = log P(b|a).")
if ready("language score"):
    try:
        best = None
        for seed in range(4):                                          # a few restarts: annealing can still get stuck
            k, s = anneal(CIPHER, seed=seed)
            if best is None or s > best[1]: best = (k, s)
        cracked = decrypt(CIPHER, best[0])
        acc = np.mean([a == b_ for a, b_ in zip(cracked, SECRET)])
        print("cracked message:\n", cracked); print(f"\ncharacters correct: {acc:.1%}  (score {best[1]:.0f} vs true {score(SECRET):.0f})")
        check("cipher cracked", lambda: acc > 0.9, "At least 90 % of the characters should be decoded correctly (rare letters may stay swapped).")
    except NotImplementedError:
        print("❌ cipher cracked: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def score(text):
    return float(sum(LOGP[IDX[a], IDX[c]] for a, c in zip(text, text[1:])))
```

```python
def anneal(cipher, iters=20000, T0=10.0, seed=0):
    rng = np.random.default_rng(seed)
    key = list(ALPHA); rng.shuffle(key)
    cur = score(decrypt(cipher, key)); best, best_key = cur, key[:]
    for it in range(iters):
        T = T0 * (1 - it / iters) + 1e-3
        i, j = rng.choice(len(key), 2, replace=False)
        cand = key[:]; cand[i], cand[j] = cand[j], cand[i]
        s = score(decrypt(cipher, cand))
        if s > cur or rng.random() < np.exp((s - cur) / T):
            key, cur = cand, s
            if cur > best: best, best_key = cur, key[:]
    return best_key, best
```
</details>

## Stretch goals
- Trigram language model: does it crack shorter messages?
- How short can the message be before it fails? Plot accuracy vs. message length.
- Encrypt your own Danish sentence and send it to a friend with the notebook.